# zlc-data role-axis tutorial

This tutorial uses only synthetic NumPy data. Each small section teaches one data capability and prints the object or result that it produces. Invalid-input guards live in `tests/`, so this notebook stays readable and exploratory.

## 0. Import the package

The package is intentionally independent: the tutorial imports the role-axis API and reports where Python resolved it.

In [1]:
from pathlib import Path
import tempfile

import numpy as np
import zlc_data
from zlc_data import (
    AxisId, AxisSpec, BlockId, DatasetRevisionRef, DatasetSchema,
    GridTopology, PointColumn, PointTable, REPEAT, SCAN_POINT,
    SPATIAL_X, SPATIAL_Y, StreamGenerationId, ValidityContract,
    ValueSchema, Selection, DataTransformSpec, apply_transform,
    commit_transform, expand_snapshot_validity, load_npz,
    owned_snapshot_from_arrays, resolve_transformed_schema, save_npz,
)
from zlc_data.codec import dataset_schema_from_tree, dataset_schema_to_tree
from zlc_data.transform_codec import committed_transform_from_tree, committed_transform_to_tree
from zlc_data.value import dataset_cell_value

print("zlc_data version:", zlc_data.__version__)
print("resolved package:", Path(zlc_data.__file__).resolve())

zlc_data version: 0.1.0
resolved package: C:\Users\eadri\Dropbox\WorkCode\Github\zlc_data\src\zlc_data\__init__.py


## The checks every package shares

A dataset is only as trustworthy as the values that went into it, so the
same handful of checks guards every boundary in the project: text that must
be canonical, numbers that must be finite, counts that must be positive.
They live here because the data contract is what they defend, and they are
declared here because a sibling package reaching into a submodule for one
is the same surface with the declaration skipped.


In [2]:
import numpy as np
from zlc_data import (canonical_text, digest_text, exact_mapping, finite_real,
                      integer, nonnegative_integer, positive_integer, take_indices,
                      expand_component_validity, AxisId, AxisSpec, SITE, VALID,
                      ValidityContract, ValidityMode, ValueSchema)

print('text   :', canonical_text('cooling', 'signal name'))
# digest_text CHECKS a content name rather than making one: the width IS
# the contract, so a digest of the wrong size is refused at the boundary.
print('digest :', digest_text('0123456789abcdef0123456789abcdef', 'schema name'))
print('numbers:', finite_real(1.5, 'exposure'), integer(3, 'frames'),
      nonnegative_integer(0, 'offset'), positive_integer(2, 'repeat'))
print('mapping:', exact_mapping({'schema': 'gain.v1', 'value': 1.0},
                               {'schema', 'value'}, 'gain.v1'))
# A contiguous selection is a slice, and a slice copies nothing.
cut = take_indices(np.arange(24).reshape(2, 3, 4), range(1, 3), axis=2)
print('take   :', cut.shape, '| a view:', cut.base is not None)
# One verdict for a cell, spread over every component that cell declares.
site = AxisSpec(AxisId('site'), 'site', SITE, 3)
schema = ValueSchema((site,), ValidityContract(ValidityMode.COMPONENTS, (site.axis_id,)),
                     np.dtype('float64'))
print('valid  :', expand_component_validity(VALID, schema))


text   : cooling
digest : 0123456789abcdef0123456789abcdef
numbers: 1.5 3 0 2
mapping: {'schema': 'gain.v1', 'value': 1.0}
take   : (2, 3, 2) | a view: True
valid  : [ True  True  True]


## 1. Describe the role-axis domain

A repeat axis is structural. Point rows hold authored scan coordinates, while `GridTopology` records the logical grid mapping.

In [3]:
from zlc_data import AxisRoleId, COMPONENT, READOUT_EVENT, SITE
repeat_axis = AxisSpec(
    AxisId("capture.repeat"), "capture.repeat", REPEAT, 2, (0, 1)
)
detuning_axis = AxisSpec(
    AxisId("scan.detuning"), "scan.detuning", SCAN_POINT,
    3, (0.0, 1.0, 2.0), unit="MHz"
)
point_table = PointTable(3, (PointColumn(
    detuning_axis.axis_id, detuning_axis.name, detuning_axis.role,
    PointColumn.NUMERIC, detuning_axis.coordinates, unit=detuning_axis.unit,
),))
grid_topology = GridTopology(
    (detuning_axis.axis_id,), ((0.0, 1.0, 2.0),), ((0,), (1,), (2,))
)
role_examples = tuple(
    AxisSpec(AxisId("role." + name), name, role, 1, (0,))
    for name, role in (("component", COMPONENT),
                       ("readout", READOUT_EVENT), ("site", SITE))
)
print("repeat size:", repeat_axis.size)
print("point rows:", point_table.row_count, point_table.columns[0].values)
print("grid dimensions:", grid_topology.dimension_ids, grid_topology.logical_shape)
print("extra roles:", [(axis.role.value, isinstance(axis.role, AxisRoleId)) for axis in role_examples])

repeat size: 2
point rows: 3 (0, 1, 2)
grid dimensions: (AxisId(value='scan.detuning'),) (3,)
extra roles: [('component', True), ('readout-event', True), ('site', True)]


## 2. Add the cell schema

The cell schema names the physical image axes, records canonical units, and declares that validity may vary by image x component.

In [4]:
from zlc_data import CoordinateFrameId
camera_frame = CoordinateFrameId("camera-pixels")
image_y = AxisSpec(
    AxisId("camera.image.y"), "camera.image.y", SPATIAL_Y,
    3, (0, 1, 2), unit="pixel", coordinate_frame=camera_frame
)
image_x = AxisSpec(
    AxisId("camera.image.x"), "camera.image.x", SPATIAL_X,
    4, (0, 1, 2, 3), unit="pixel", coordinate_frame=camera_frame
)
cell_schema = ValueSchema(
    (image_y, image_x), ValidityContract.components(image_x.axis_id),
    np.dtype("<f4"), value_unit="count"
)
schema = DatasetSchema(repeat_axis, point_table, grid_topology, cell_schema)
print("data axes:", [axis.axis_id.value for axis in cell_schema.data_axes])
print("physical shape:", schema.physical_shape)
print("value unit:", cell_schema.value_unit, "axis units:", image_x.unit)

data axes: ['camera.image.y', 'camera.image.x']
physical shape: (2, 3, 3, 4)
value unit: count axis units: pixel


## 3. Construct an immutable snapshot from arrays

`owned_snapshot_from_arrays` accepts a complete schema and a dense validity mask. It freezes the values, computes the schema-bound reference, and compacts validity under the declared contract.

In [5]:
from zlc_data import DataBlock, DatasetRevision, OwnedSnapshot
values = np.arange(np.prod(schema.physical_shape), dtype="<f4").reshape(schema.physical_shape)
dense_validity = np.ones(schema.physical_shape, dtype=bool)
dense_validity[0, 2, :, 1] = False
snapshot = owned_snapshot_from_arrays(
    schema=schema, values=values, revision=DatasetRevision(0), validity=dense_validity,
    block_id=BlockId("usage-block"),
    stream_generation=StreamGenerationId("usage-generation"),
)
print("snapshot ref:", snapshot.ref)
print("values shape/dtype:", snapshot.block.values.shape, snapshot.block.values.dtype)
print("stored validity carrier:", type(snapshot.block.validity).__name__)
print("immutable carriers:", isinstance(snapshot, OwnedSnapshot), isinstance(snapshot.block, DataBlock))

snapshot ref: DatasetRevisionRef(block_id=BlockId(value='usage-block'), stream_generation=StreamGenerationId(value='usage-generation'), schema_fingerprint='1c9763ed7a7f828ad9dde3be4ecaea26', revision=DatasetRevision(value=0))
values shape/dtype: (2, 3, 3, 4) float32
stored validity carrier: DatasetComponentValidity
immutable carriers: True True


## 4. Expand named validity for inspection

The stored validity is compact and named. Expand it only when a consumer needs a mask aligned to `(repeat, point, *data_axes)`.

In [6]:
from zlc_data import (
    CellValidity, DatasetComponentValidity, INVALID, Invalid, VALID, Valid,
    ValidityMode, compact_dataset_validity, expand_dataset_validity,
)
expanded_validity = expand_snapshot_validity(snapshot)
cell_validity = CellValidity(np.array([[True, True, True], [True, False, True]]))
cell_dense = expand_dataset_validity(cell_validity, schema)
cell_round_trip = compact_dataset_validity(cell_dense, schema)
invalid_positions = np.argwhere(~expanded_validity)
print("expanded shape:", expanded_validity.shape)
print("first invalid positions:", invalid_positions[:5].tolist())
print("mask writeable:", expanded_validity.flags.writeable)
print("cell validity round-trip:", type(cell_round_trip).__name__, np.array_equal(
    expand_dataset_validity(cell_round_trip, schema), cell_dense
))
print("dataset component carrier:", isinstance(snapshot.block.validity, DatasetComponentValidity))
print("marker types:", isinstance(VALID, Valid), isinstance(INVALID, Invalid))
print("contract modes:", schema.cell_schema.validity_contract.mode is ValidityMode.COMPONENTS,
      ValidityContract.value().mode is ValidityMode.VALUE)

expanded shape: (2, 3, 3, 4)
first invalid positions: [[0, 2, 0, 1], [0, 2, 1, 1], [0, 2, 2, 1]]
mask writeable: False
cell validity round-trip: DatasetComponentValidity True
dataset component carrier: True
marker types: True True
contract modes: True True


## 5. Commit and resolve a transform

A selection is first committed against the source schema. The resolved schema tells you the output shape before values are touched.

In [7]:
from zlc_data import CommittedTransform
spec = DataTransformSpec((
    Selection.index(repeat_axis.axis_id, 0),
    Selection.index_range(image_x.axis_id, 1, 4),
))
committed = commit_transform(schema, spec)
resolved_schema = resolve_transformed_schema(schema, committed)
print("operations:", [type(item).__name__ for item in spec.operations])
print("source/output shapes:", schema.physical_shape, resolved_schema.physical_shape)
print("source fingerprint:", committed.source_schema_fingerprint)
print("committed transform type:", isinstance(committed, CommittedTransform))

operations: ['Selection', 'Selection']
source/output shapes: (2, 3, 3, 4) (1, 3, 3, 3)
source fingerprint: 1c9763ed7a7f828ad9dde3be4ecaea26
committed transform type: True


## 6. Apply the committed transform

Applying the committed operation to the owned snapshot returns transformed values together with the effective output schema and validity.

In [8]:
transformed = apply_transform(snapshot, committed)
print("transformed shape:", transformed.values.shape)
print("effective schema matches:", transformed.schema == committed.effective_output_schema)
print("transformed validity shape:", transformed.expanded_validity().shape)

transformed shape: (1, 3, 3, 3)
effective schema matches: True
transformed validity shape: (1, 3, 3, 3)


## 7. Reduce a named source axis

`AxisSourceRef` makes the reduction domain explicit; the committed result records the method and the resulting schema.

In [9]:
from zlc_data import AxisSourceRef, ReductionMethod, ReductionSpec
reduction_spec = ReductionSpec(
    (AxisSourceRef.tensor(repeat_axis.axis_id),), ReductionMethod.MEAN
)
reduction_transform = commit_transform(
    schema, DataTransformSpec((reduction_spec,))
)
reduced = apply_transform(snapshot, reduction_transform)
print("reduction method/sources:", reduction_spec.method.value, reduction_spec.sources)
print("reduced shape/dtype:", reduced.values.shape, reduced.values.dtype)
print("reduced repeat size:", reduced.schema.repeat_axis.size)

reduction method/sources: MEAN (AxisSourceRef(kind='TENSOR', axis_id=AxisId(value='capture.repeat')),)
reduced shape/dtype: (1, 3, 3, 4) float64
reduced repeat size: 1


## 8. Histogram a tensor axis

A histogram is also a committed transform: its frozen edges become the coordinates of a new histogram-bin axis.

In [10]:
from zlc_data import HistogramSpec
histogram_spec = HistogramSpec(
    (AxisSourceRef.tensor(image_x.axis_id),), (0.0, 24.0, 48.0, 72.0)
)
histogram_transform = commit_transform(
    schema, DataTransformSpec((histogram_spec,))
)
histogrammed = apply_transform(snapshot, histogram_transform)
print("bin edges/id:", histogram_spec.bin_edges, histogram_spec.bin_axis_id)
print("histogram shape/dtype:", histogrammed.values.shape, histogrammed.values.dtype)
print("bin coordinates:", histogrammed.schema.cell_schema.data_axes[-1].coordinates)

bin edges/id: (0.0, 24.0, 48.0, 72.0) zlc_data.histogram-bin
histogram shape/dtype: (2, 3, 3, 3) int64
bin coordinates: (12, 36, 60)


## 9. Project one dataset cell as a typed Value

`dataset_cell_value` keeps the cell schema and projects the dataset-level validity into the Value-level contract.

In [11]:
from zlc_data import ComponentValidity, Value, ValuePayloadContract
component_example = ComponentValidity(
    (image_x.axis_id,), np.ones((image_x.size,), dtype=bool)
)
typed_value = Value(np.zeros(cell_schema.data_shape, dtype=cell_schema.dtype),
                    component_example, cell_schema)
ValuePayloadContract(cell_schema).validate(typed_value)
value = dataset_cell_value(snapshot.block, repeat_index=0, point_ordinal=0)
print("explicit Value contract:", isinstance(typed_value, Value),
      type(typed_value.validity).__name__)
print("value shape/dtype:", value.values.shape, value.values.dtype)
print("value schema axes:", [axis.axis_id.value for axis in value.schema.data_axes])
print("value validity:", type(value.validity).__name__)

explicit Value contract: True ComponentValidity
value shape/dtype: (3, 4) float32
value schema axes: ['camera.image.y', 'camera.image.x']
value validity: ComponentValidity


## 10. Materialize that Value as a single-cell dataset

A projection helper gives a Value a new dataset carrier while retaining the source revision and using a new derived block identity.

In [12]:
from zlc_data import materialize_derived_dataset, materialize_scalar_dataset, materialize_value_dataset
def reference_for(block_name):
    def build(derived_schema):
        return DatasetRevisionRef(
            BlockId(block_name), snapshot.ref.stream_generation,
            derived_schema.fingerprint, snapshot.ref.revision,
        )
    return build

derived = materialize_value_dataset(
    snapshot.ref, value, reference_for=reference_for("usage-derived")
)
full_derived = materialize_derived_dataset(
    snapshot.ref, snapshot.block.values + 1, schema=schema, validity=VALID,
    reference_for=reference_for("usage-derived-full")
)
scalar = materialize_scalar_dataset(
    snapshot.ref, np.float32(1.5), valid=True, unit="count",
    reference_for=reference_for("usage-derived-scalar")
)
print("derived shape:", derived.block.schema.physical_shape)
print("derived block:", derived.ref.block_id)
print("derived validity:", type(derived.block.validity).__name__)
print("full/scalar shapes:", full_derived.block.values.shape, scalar.block.values.shape)

derived shape: (1, 1, 3, 4)
derived block: usage-derived
derived validity: DatasetComponentValidity
full/scalar shapes: (2, 3, 3, 4) (1, 1, 1)


## 11. Encode and decode schema and transform contracts

Tree codecs make the role-axis schema and committed transform explicit, canonical, and suitable for strict persistence or transport.

In [13]:
schema_tree = dataset_schema_to_tree(schema)
schema_copy = dataset_schema_from_tree(schema_tree)
print("schema tree fields:", sorted(schema_tree))
print("schema round-trip:", schema_copy == schema)
print("fingerprint:", schema_copy.fingerprint)

schema tree fields: ['cell_schema', 'grid_topology', 'point_table', 'repeat_axis', 'schema']
schema round-trip: True
fingerprint: 1c9763ed7a7f828ad9dde3be4ecaea26


### Transform codec

The committed form carries the source fingerprint, exact point scope, specification, and effective output schema.

In [14]:
transform_tree = committed_transform_to_tree(committed)
transform_copy = committed_transform_from_tree(transform_tree)
print("transform tree fields:", sorted(transform_tree))
print("transform round-trip:", transform_copy == committed)
print("exact point scope:", transform_copy.exact_point_ordinals)

transform tree fields: ['effective_output_schema', 'exact_point_ordinals', 'schema', 'source_schema_fingerprint', 'spec']
transform round-trip: True
exact point scope: (0, 1, 2)


## 12. Persist and restore the owned snapshot

NPZ persistence stores the typed manifest and arrays without pickle. The temporary file is removed automatically after this demonstration.

In [15]:
with tempfile.TemporaryDirectory() as directory:
    path = Path(directory) / "usage-snapshot.npz"
    save_npz(path, snapshot)
    restored_snapshot = load_npz(path)
    print("archive exists during round-trip:", path.exists())
    print("snapshot exactly equal:", snapshot.exactly_equals(restored_snapshot))
    print("dense validity equal:", np.array_equal(
        snapshot.expanded_validity(), restored_snapshot.expanded_validity()
    ))

archive exists during round-trip: True
snapshot exactly equal: True
dense validity equal: True


## 12. See the strict persistence boundary

A missing or malformed archive is reported as `NPZFormatError`, so callers do not have to interpret backend-specific exceptions.

In [16]:
from zlc_data import NPZFormatError
with tempfile.TemporaryDirectory() as directory:
    missing_path = Path(directory) / "missing.npz"
    try:
        load_npz(missing_path)
    except NPZFormatError as error:
        print("missing archive error:", type(error).__name__)

missing archive error: NPZFormatError


## 13. Use the scalar validation boundary

The data layer owns small canonical validators for boundaries such as identifiers, finite numbers, and non-negative integers. Detailed rejection cases are tested in `tests/`.

In [17]:
from zlc_data import canonical_text, finite_real, nonnegative_integer

print("canonical text:", canonical_text("camera.frame", "name"))
print("finite real:", finite_real(3.5, "gain"))


canonical text: camera.frame
finite real: 3.5


## 14. Say which rows were selected, and write a snapshot down

A selection is published by one package and consumed by another, so what one IS -- and what a
change to one means -- belongs here rather than in either end.  `snapshot_manifest` /
`snapshot_from_manifest` are the same story for storage: an archive of a snapshot is the arrays
plus the manifest that says what they are, and both halves have to come from one place.


In [18]:
from zlc_data import (
    IndexSelection,
    SelectionChange,
    resolve_selection_indices,
    snapshot_from_manifest,
    is_intrinsically_immutable_array,
    snapshot_manifest,
)

print('what a change can be:', [member.value for member in SelectionChange])
repeat_axis = snapshot.block.schema.repeat_axis
picked = IndexSelection(axis_id=repeat_axis.axis_id, index=0)
print('rows it names:', resolve_selection_indices(repeat_axis, picked))
stored = {}
manifest = snapshot_manifest(snapshot, stored, values_key='frames')
print('manifest keys:', sorted(manifest))
print('frozen array?', is_intrinsically_immutable_array(manifest and stored['frames']))
print('read back:', type(snapshot_from_manifest(manifest, stored)).__name__)


what a change can be: ['added', 'updated', 'committed', 'removed']
rows it names: (range(0, 1), True)
manifest keys: ['format', 'ref', 'schema', 'validity', 'values_key', 'version']
frozen array? True
read back: OwnedSnapshot
